#### MILE -- Multi-Chain (K unabhaengige Chains x S Samples pro Chain)

Erweitert `02_mile_warmup_extended.ipynb` (Single-Chain, `N_CHAINS=1`) um echte Chain-Diversitaet: jede Chain startet nicht vom selben `theta_map`, sondern vom `theta_map` eines **unabhaengig trainierten Ensemble-Mitglieds** (Seeds 2-6, aus `01_train_ensemble_member.ipynb`). Das ist die eigentliche "Ensemble"-Komponente von MILE (Microcanonical Langevin **Ensembles**) -- die K*S-Formel aus dem Paper (Appendix A.4) mittelt explizit ueber K Chains UND S Samples pro Chain. Bisher hatten wir nur die S-Achse (eine Chain, mehrere Samples); dieses Notebook nutzt zusaetzlich die K-Achse, mit K = Anzahl verfuegbarer Ensemble-Checkpoints.

Gleiche Posterior-Daten (512 Beispiele) und gleiche MCMC-Konfiguration pro Chain wie im Single-Chain-Lauf, damit der Vergleich sauber bleibt.


In [1]:
from pathlib import Path

# ============================================================
# CONFIG
# MILE -- MULTI-CHAIN (K Chains x S Samples/Chain)
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"
MILE_REPO = MASTER_DIR / "MILE"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)

RESULT_DIR = MASTER_DIR / "method_results" / "mile_qwen_agnews"

# ------------------------------------------------------------
# Chains -- ein unabhaengig trainierter Checkpoint pro Chain.
# Nur Seeds aufnehmen, die schon als Checkpoint vorliegen
# (~/Masterarbeit/checkpoints/qwen_lora_agnews_seed{SEED}/) --
# einfach die Liste anpassen, wenn noch nicht alle fertig sind.
# ------------------------------------------------------------

# Standardwerte greifen beim interaktiven Ausfuehren wie bisher.
# Fuer SLURM-Batch-Jobs koennen diese drei Werte per Umgebungsvariable
# ueberschrieben werden (z.B. CHAIN_SEEDS='[5]' vor dem Aufruf setzen),
# ohne das Notebook zu duplizieren.
import os
import json as _json

CHAIN_SEEDS = _json.loads(os.environ.get("CHAIN_SEEDS", "[1, 2, 3]"))
WARMUP_STEPS_OVERRIDE = os.environ.get("WARMUP_STEPS")
N_SAMPLES_PER_CHAIN_OVERRIDE = os.environ.get("N_SAMPLES_PER_CHAIN")

CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"

# ------------------------------------------------------------
# Posterior data (identisch zum Single-Chain-Lauf)
# ------------------------------------------------------------

N_PER_CLASS = int(os.environ.get("N_PER_CLASS", "512"))        # erweitert von 128 -- 512 x 4 = 2048 examples
SEQ_LEN = 32
POSTERIOR_KEY_SEED = 2027

# ------------------------------------------------------------
# Prior (identisch zum Single-Chain-Lauf)
# ------------------------------------------------------------

PRIOR_STD = 0.02

# ------------------------------------------------------------
# MCLMC -- automatische Adaption, PRO Chain identische Config
# wie im Single-Chain-Lauf (02_mile_warmup_extended.ipynb)
# ------------------------------------------------------------

WARMUP_STEPS = int(WARMUP_STEPS_OVERRIDE) if WARMUP_STEPS_OVERRIDE else 250
N_SAMPLES_PER_CHAIN = int(N_SAMPLES_PER_CHAIN_OVERRIDE) if N_SAMPLES_PER_CHAIN_OVERRIDE else 15
THINNING_STEPS = 30
N_THINNING = 1

DIAGONAL_PRECONDITIONING = False
DESIRED_ENERGY_VAR_START = 0.0005
DESIRED_ENERGY_VAR_END = 0.0001
TRUST_IN_ESTIMATE = 1.5
NUM_EFFECTIVE_SAMPLES = 10
STEP_SIZE_INIT = 1e-5

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64
EVAL_MODE = "test"   # "test" -> 7600 echte Testbeispiele (finaler Report)
                     # "val"  -> 12000 Val-Beispiele (falls je fuer Tuning gebraucht)

# ------------------------------------------------------------
# Automatic run name
# ------------------------------------------------------------

N_POSTERIOR_CONFIG = 4 * N_PER_CLASS
N_CHAINS = len(CHAIN_SEEDS)
N_SAMPLES_TOTAL = N_CHAINS * N_SAMPLES_PER_CHAIN

RUN_NAME = (
    f"mile_multichain_k{N_CHAINS}"
    f"_balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_warmup{WARMUP_STEPS}"
    f"_spc{N_SAMPLES_PER_CHAIN}"
    f"_thin{THINNING_STEPS}"
    f"_{EVAL_MODE}eval"
)

print("======================================")
print("MILE -- MULTI-CHAIN")
print("======================================")
print("Run name:              ", RUN_NAME)
print("Chain seeds:           ", CHAIN_SEEDS)
print("K (chains):            ", N_CHAINS)
print("S (samples/chain):     ", N_SAMPLES_PER_CHAIN)
print("Total samples (K*S):   ", N_SAMPLES_TOTAL)
print("Examples/class:        ", N_PER_CLASS)
print("Total posterior ex.:   ", N_POSTERIOR_CONFIG)
print("Prior std:             ", PRIOR_STD)
print("Warmup steps/chain:    ", WARMUP_STEPS)
print("Thinning steps/sample: ", THINNING_STEPS)
print("Eval mode:             ", EVAL_MODE)
print("======================================")


MILE -- MULTI-CHAIN
Run name:               mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval
Chain seeds:            [2]
K (chains):             1
S (samples/chain):      30
Total samples (K*S):    30
Examples/class:         2048
Total posterior ex.:    8192
Prior std:              0.02
Warmup steps/chain:     250
Thinning steps/sample:  30
Eval mode:              test


In [2]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import json
import runpy
import time

import numpy as np
import jax
import jax.numpy as jnp

from jax import random

RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(MILE_REPO) not in sys.path:
    sys.path.insert(0, str(MILE_REPO))

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert MILE_REPO.exists()
assert BASELINE_SCRIPT.exists()

NOTEBOOK_START_TIME = time.time()  # Gesamtlaufzeit ab hier gemessen

for seed in CHAIN_SEEDS:
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    found = list(seed_dir.glob("*_pretrained_params.npy")) if seed_dir.exists() else []
    status = "OK" if len(found) == 1 else f"PROBLEM ({len(found)} files)"
    print(f"Seed {seed}: {seed_dir} -> {status}")
    assert len(found) == 1, (
        f"Erwarte genau 1 Checkpoint fuer Seed {seed} in {seed_dir}, "
        f"gefunden: {len(found)}. CHAIN_SEEDS anpassen oder Checkpoint nachliefern."
    )


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3


Devices: [CudaDevice(id=0)]
Seed 2: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed2 -> OK


In [3]:
# ============================================================
# ENVIRONMENT-CHECK (selbstheilend) -- installiert fehlende Pakete
# automatisch per `uv add` nach (NICHT pip -- das .venv hat kein pip,
# uv verwaltet Pakete komplett selbst). blackjax explizit auf 1.2.2
# gepinnt (MILE/pyproject.toml verlangt genau diese Version -- neuere
# Versionen wie 1.2.5 haben `streaming_average_update` aus
# blackjax.util entfernt/verschoben, MILEs eigener Code bricht damit).
# ============================================================

import subprocess
import os

UV_PATH = os.path.expanduser("~/.local/bin/uv")


def uv_add(package_name):
    print(f"uv add {package_name} (in {QWEN_REPO}) ...")
    result = subprocess.run(
        [UV_PATH, "add", package_name],
        cwd=str(QWEN_REPO),
        capture_output=True, text=True,
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f"uv add {package_name} fehlgeschlagen.")
    print(f"'{package_name}' hinzugefuegt.")


MAX_ATTEMPTS = 5

for attempt in range(MAX_ATTEMPTS):
    try:
        import blackjax
        from src.config.sampler import SamplerConfig, Sampler
        from src.training.sampling import warmup_mclmc
        break
    except ModuleNotFoundError as exc:
        missing = exc.name
        print(f"Fehlendes Modul erkannt: '{missing}' (Versuch {attempt + 1}/{MAX_ATTEMPTS})")
        uv_add("blackjax==1.2.2" if missing == "blackjax" else missing)
else:
    raise RuntimeError(
        f"Nach {MAX_ATTEMPTS} Versuchen immer noch fehlende Module -- manuell pruefen."
    )

print("blackjax:", blackjax.__version__)
print("MILE-Sampler-Module: OK")


blackjax: 1.2.2
MILE-Sampler-Module: OK


In [4]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [5]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (einmal -- fuer env/data und die
# LoRA-Struktur, die fuer alle Chains identisch ist, da nur die
# LoRA-Werte sich zwischen den Seeds unterscheiden, nicht die
# eingefrorene Backbone-Struktur)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]


Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
2026-09-10 12:15:07,514 - root - INFO - Using subspace model: lora_category


2026-09-10 12:15:07,515 - subspace_inference.curve_optimizer.subspace_curve - INFO - Using LoRA Mixin with rank=8, dtype=float32, alpha=16.0, mode=A(t)eB(t), rho=0.25, rho_s=0.0


2026-09-10 12:15:07,515 - subspace_inference.curve_optimizer.subspace_curve - INFO - Using Bezier curve parameterization


2026-09-10 12:15:08,015 - subspace_inference.curve_optimizer.subspace_curve - WARNING - Using LoRA mode A(t)eB(t) with lora_rho_s=0 makes no sense and is equivalent to A(t)B(t).


...

Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246199
val_acc: 0.913667
val_ece: 0.009229
val_brier: 0.131579
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251077
test_acc: 0.914737
test_ece: 0.007497
test_brier: 0.131565
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [6]:
# ============================================================
# POSTERIOR SETUP (geteilt ueber alle Chains -- gleiche 512
# Beispiele, gleicher Log-Posterior/rebuild_full_params fuer alle
# Chains; nur der jeweilige Chain-Startpunkt theta_map ist anders)
# ============================================================

posterior = qpu.setup_qwen_posterior(
    env=env,
    params=params,
    data=data,
    n_per_class=N_PER_CLASS,
    seq_len=SEQ_LEN,
    posterior_key_seed=POSTERIOR_KEY_SEED,
    prior_std=PRIOR_STD,
)

x_posterior = posterior.x_posterior
y_posterior = posterior.y_posterior
posterior_example_keys = posterior.posterior_example_keys
N_POSTERIOR_EXAMPLES = posterior.n_posterior_examples
subset_indices_host = posterior.subset_indices_host

rebuild_full_params = posterior.rebuild_full_params
qwen_log_posterior = posterior.qwen_log_posterior

# theta_map des Baseline-Checkpoints (Seed aus BASELINE_SCRIPT, i.d.R.
# Seed 2) -- dient hier nur als Referenzpunkt fuer Distanz-Diagnostik,
# NICHT als gemeinsamer Chain-Startpunkt (jede Chain nutzt ihren
# eigenen, unabhaengig trainierten theta_map, siehe unten).
reference_theta_map = posterior.theta_map


Subset indices: [   0    1    4 ... 8128 8131 8135]
Labels: [0 0 0 ... 3 3 3]
Number of examples: 8192
input_ids: (8192, 32)
attention_mask: (8192, 32)
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672


All MAP parameters finite: True


In [7]:
# ============================================================
# CHAIN-SPEZIFISCHE theta_map LADEN (ein unabhaengig trainiertes
# Ensemble-Mitglied pro Chain -- gleiche Vorgehensweise wie in
# 05_deep_ensemble.ipynb: parametrisierte Kopie von
# evaluate_saved_baseline.py mit anderem CHECKPOINT_PATH/SEED)
# ============================================================

_eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")


def load_chain_theta_map(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_path = list(seed_dir.glob("*_pretrained_params.npy"))[0]

    old_checkpoint_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("CHECKPOINT_PATH")
    )
    old_seed_line = next(
        line for line in _eval_script_template.splitlines()
        if line.strip().startswith("SEED")
    )

    script_text = _eval_script_template.replace(
        old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(
        old_seed_line, f"SEED = {seed}",
    )

    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora"
        / f"_tmp_eval_chain_seed{seed}.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        chain_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)

    chain_theta_map, *_ = qpu.vectorize_lora_params(
        chain_objects["params"]["params"]
    )
    return chain_theta_map


chain_theta_maps = {}
for seed in CHAIN_SEEDS:
    print(f"Loading theta_map for chain seed={seed}...")
    chain_theta_maps[seed] = load_chain_theta_map(seed)

print()
print("Loaded", len(chain_theta_maps), "chain starting points.")
for seed, theta in chain_theta_maps.items():
    distance = float(jnp.linalg.norm(theta - reference_theta_map))
    print(f"  seed={seed}  ||theta - reference_theta_map|| = {distance:.4f}")


Loading theta_map for chain seed=2...


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
2026-09-10 12:17:15,196 - root - INFO - Using subspace model: lora_category


2026-09-10 12:17:15,196 - subspace_inference.curve_optimizer.subspace_curve - INFO - Using LoRA Mixin with rank=8, dtype=float32, alpha=16.0, mode=A(t)eB(t), rho=0.25, rho_s=0.0


2026-09-10 12:17:15,197 - subspace_inference.curve_optimizer.subspace_curve - INFO - Using Bezier curve parameterization


2026-09-10 12:17:15,199 - subspace_inference.curve_optimizer.subspace_curve - WARNING - Using LoRA mode A(t)eB(t) with lora_rho_s=0 makes no sense and is equivalent to A(t)B(t).


...

post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246199
val_acc: 0.913667
val_ece: 0.009229
val_brier: 0.131579
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251077
test_acc: 0.914737
test_ece: 0.007497
test_brier: 0.131565
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True



Loaded 1 chain starting points.
  seed=2  ||theta - reference_theta_map|| = 0.0000


In [8]:
# ============================================================
# MCLMC WARMUP + SAMPLING (wiederverwendbar pro Chain -- exakt
# dieselbe Logik wie in 02_mile_warmup_extended.ipynb, nur als
# Funktion gekapselt und ueber den Chain-Startpunkt parametrisiert)
# ============================================================

from src.config.sampler import SamplerConfig, Sampler
from src.training.sampling import warmup_mclmc

sampler_config = SamplerConfig(
    name=Sampler.MCLMC,
    n_chains=1,
    warmup_steps=WARMUP_STEPS,
    n_samples=N_SAMPLES_PER_CHAIN,
    n_thinning=N_THINNING,
    diagonal_preconditioning=DIAGONAL_PRECONDITIONING,
    desired_energy_var_start=DESIRED_ENERGY_VAR_START,
    desired_energy_var_end=DESIRED_ENERGY_VAR_END,
    trust_in_estimate=TRUST_IN_ESTIMATE,
    num_effective_samples=NUM_EFFECTIVE_SAMPLES,
    step_size_init=STEP_SIZE_INIT,
    keep_warmup=False,
)


def run_one_chain(chain_theta_map, rng_key, chain_label):
    """Adaptive MCLMC-Warmup + Thinned Sampling ab chain_theta_map.
    Identische Logik wie im Single-Chain-Notebook. Gibt
    (sample_positions, tuned_parameters, rng_key) zurueck."""
    rng_key, warmup_key, sampling_key = random.split(rng_key, 3)

    warmup_state, tuned_parameters = warmup_mclmc(
        config=sampler_config,
        rng_key=warmup_key,
        init_params=chain_theta_map,
        unnorm_log_posterior=qwen_log_posterior,
        n_devices=1,
    )
    warmup_state = jax.block_until_ready(warmup_state)

    warmup_position_host = np.asarray(jax.device_get(warmup_state.position))
    warmup_is_finite = bool(np.isfinite(warmup_position_host).all())

    print(f"[{chain_label}] warmup finite: {warmup_is_finite}")
    for name, value in tuned_parameters.items():
        value_host = np.asarray(jax.device_get(value))
        print(f"[{chain_label}]   tuned {name}: {value_host}")

    assert warmup_is_finite, (
        f"[{chain_label}] Warmup enthaelt NaN/Inf -- Chain abgebrochen."
    )

    mclmc_sampler = sampler_config.kernel(qwen_log_posterior, **tuned_parameters)

    def one_thinned_sample(state, keys):
        def inner_step(inner_state, key):
            new_state, info = mclmc_sampler.step(key, inner_state)
            return new_state, None
        state, _ = jax.lax.scan(inner_step, state, keys)
        return state, state.position

    rng_key, sampling_key = random.split(rng_key)
    sampling_keys = random.split(
        sampling_key, N_SAMPLES_PER_CHAIN * THINNING_STEPS,
    ).reshape(N_SAMPLES_PER_CHAIN, THINNING_STEPS, -1)

    final_state, sample_positions = jax.lax.scan(
        one_thinned_sample, warmup_state, sampling_keys,
    )
    final_state = jax.block_until_ready(final_state)
    sample_positions = jax.block_until_ready(sample_positions)

    samples_host = np.asarray(jax.device_get(sample_positions))
    finite_per_sample = np.isfinite(samples_host).all(axis=1)

    print(f"[{chain_label}] sampling done. All finite: {finite_per_sample.all()}")
    assert finite_per_sample.all(), f"[{chain_label}] sample contains NaN/Inf."

    return sample_positions, tuned_parameters, rng_key


In [9]:
# ============================================================
# CHAIN-CACHE (K-unabhaengig!) -- die Zwischenspeicherung pro Chain
# darf NICHT von N_CHAINS/RUN_NAME abhaengen, sonst werden bereits
# fertige Chains beim Aendern von CHAIN_SEEDS (z.B. [1,2] -> [1,2,3])
# faelschlich neu berechnet, weil sich der Dateiname aendert.
# Der Cache-Key enthaelt nur, was eine Chain-Berechnung tatsaechlich
# eindeutig macht: Seed, Posterior-Daten, Prior, MCMC-Konfiguration.
# ============================================================

CHAIN_CACHE_KEY = (
    f"balanced{N_POSTERIOR_CONFIG}"
    f"_seq{SEQ_LEN}"
    f"_prior{PRIOR_STD}"
    f"_warmup{WARMUP_STEPS}"
    f"_spc{N_SAMPLES_PER_CHAIN}"
    f"_thin{THINNING_STEPS}"
    f"_{EVAL_MODE}eval"
)


def chain_cache_paths(seed):
    samples_path = RESULT_DIR / f"mile_chain_seed{seed}_{CHAIN_CACHE_KEY}_partial_samples.npy"
    meta_path = RESULT_DIR / f"mile_chain_seed{seed}_{CHAIN_CACHE_KEY}_partial_meta.json"
    return samples_path, meta_path


# Einmalige Migration: alte, K-abhaengige Dateinamen (aus frueheren
# Laeufen mit anderem N_CHAINS/RUN_NAME) auf das neue, K-unabhaengige
# Schema umbenennen -- damit bereits fertige Chains nicht verloren
# gehen, nur weil sich CHAIN_SEEDS/N_CHAINS geaendert hat.
for _seed in CHAIN_SEEDS:
    _new_samples_path, _new_meta_path = chain_cache_paths(_seed)
    if _new_samples_path.exists():
        continue
    _legacy_matches = sorted(RESULT_DIR.glob(
        f"mile_multichain_k*_{CHAIN_CACHE_KEY}_chain{_seed}_partial_samples.npy"
    ))
    if _legacy_matches:
        _legacy_samples_path = _legacy_matches[0]
        _legacy_meta_path = _legacy_samples_path.with_name(
            _legacy_samples_path.name.replace("_partial_samples.npy", "_partial_meta.json")
        )
        _legacy_samples_path.rename(_new_samples_path)
        _legacy_meta_path.rename(_new_meta_path)
        print(f"Migriert (altes Namensschema -> neu): seed={_seed} -> {_new_samples_path.name}")


In [10]:
# ============================================================
# ALLE CHAINS AUSFUEHREN (mit Zeitmessung + Zwischenspeicherung pro
# Chain -- bei einem Absturz/Verbindungsabbruch gehen nur noch nicht
# gespeicherte Chains verloren. Ein Neustart dieses Notebooks LAEDT
# bereits fertige Chains automatisch aus der Zwischenspeicherung,
# statt sie neu zu berechnen -- echtes Resume, kein Neustart bei 0.
# ============================================================

all_chain_sample_positions = []
all_chain_tuned_parameters = {}
chain_durations_seconds = {}

chains_start_time = time.time()

for chain_index, seed in enumerate(CHAIN_SEEDS):
    chain_label = f"chain {chain_index + 1}/{N_CHAINS} (seed={seed})"

    partial_samples_path, partial_meta_path = chain_cache_paths(seed)

    if partial_samples_path.exists() and partial_meta_path.exists():
        print(f"\n=== {chain_label} === (aus Zwischenspeicherung geladen, kein Neuberechnen)")

        chain_sample_positions = jnp.asarray(np.load(partial_samples_path))

        with open(partial_meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        chain_durations_seconds[seed] = meta["duration_seconds"]
        all_chain_tuned_parameters[seed] = meta["tuned_parameters"]

        print(f"[{chain_label}] geladen -- urspruengliche Dauer: {chain_durations_seconds[seed]:.1f}s")

    else:
        print(f"\n=== {chain_label} ===")

        chain_start_time = time.time()

        chain_sample_positions, tuned_parameters, rng_key = run_one_chain(
            chain_theta_map=chain_theta_maps[seed],
            rng_key=rng_key,
            chain_label=chain_label,
        )

        chain_duration = time.time() - chain_start_time
        chain_durations_seconds[seed] = chain_duration
        print(f"[{chain_label}] Dauer: {chain_duration:.1f}s ({chain_duration/60:.1f} min)")

        tuned_params_host = {
            name: float(np.asarray(jax.device_get(value)))
            for name, value in tuned_parameters.items()
        }
        all_chain_tuned_parameters[seed] = tuned_params_host

        # Zwischenspeichern -- ueberlebt einen Kernel-Absturz / Verbindungsabbruch
        samples_host_chain = np.asarray(jax.device_get(chain_sample_positions))
        np.save(partial_samples_path, samples_host_chain)
        with open(partial_meta_path, "w", encoding="utf-8") as f:
            json.dump({
                "seed": seed,
                "duration_seconds": chain_duration,
                "tuned_parameters": tuned_params_host,
            }, f, indent=2)
        print(f"[{chain_label}] Zwischengespeichert: {partial_samples_path}")

    all_chain_sample_positions.append(chain_sample_positions)

chains_total_duration = time.time() - chains_start_time

# K*S Samples zusammenpoolen
sample_positions = jnp.concatenate(all_chain_sample_positions, axis=0)
samples_host = np.asarray(jax.device_get(sample_positions))

print()
print("======================================")
print("ALLE CHAINS FERTIG")
print("======================================")
print("Samples pro Chain:", N_SAMPLES_PER_CHAIN)
print("Anzahl Chains:    ", N_CHAINS)
print("Samples gesamt:   ", samples_host.shape[0])
print("Dauer pro Chain (Sekunden):")
for seed, duration in chain_durations_seconds.items():
    print(f"  seed={seed}: {duration:.1f}s ({duration/60:.1f} min)")
print(f"Gesamtdauer alle Chains: {chains_total_duration:.1f}s ({chains_total_duration/60:.1f} min)")
print("Getunte Parameter pro Chain:")
for seed, tuned in all_chain_tuned_parameters.items():
    print(f"  seed={seed}: {tuned}")



=== chain 1/1 (seed=2) ===


[chain 1/1 (seed=2)] warmup finite: True
[chain 1/1 (seed=2)]   tuned step_size: 0.09324268996715546
[chain 1/1 (seed=2)]   tuned L: 0.2408163845539093


[chain 1/1 (seed=2)] sampling done. All finite: True
[chain 1/1 (seed=2)] Dauer: 19745.2s (329.1 min)
[chain 1/1 (seed=2)] Zwischengespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_chain_seed2_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval_partial_samples.npy

ALLE CHAINS FERTIG
Samples pro Chain: 30
Anzahl Chains:     1
Samples gesamt:    30
Dauer pro Chain (Sekunden):
  seed=2: 19745.2s (329.1 min)
Gesamtdauer alle Chains: 19745.3s (329.1 min)
Getunte Parameter pro Chain:
  seed=2: {'step_size': 0.09324268996715546, 'L': 0.2408163845539093}


In [11]:
# ============================================================
# BASIC SAMPLE DIAGNOSTICS (Distanz zur Referenz-MAP, seed=2)
# ============================================================

distances_from_map = qpu.compute_distances_from_map(
    reference_theta_map, sample_positions,
)


Distance summary:
Minimum: 3.583951
Mean: 5.4930124
Maximum: 6.991492


In [12]:
# ============================================================
# EVALUATION DATA (Val/Testset)
# ============================================================

if EVAL_MODE == "test":
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set (final report).")
elif EVAL_MODE == "val":
    x_eval, y_eval = data.get("val")
    print("Evaluating on the AG News VALIDATION set.")
else:
    raise ValueError(f"Unknown EVAL_MODE: {EVAL_MODE!r}")

print("Evaluation examples:", int(y_eval.shape[0]))


Evaluating on the full AG News TEST set (final report).
Evaluation examples: 7600


In [13]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES (mit Zwischenspeicherung pro
# Sample -- bei N_SAMPLES_TOTAL Samples x 7.600 Testbeispielen dauert
# dieser Schritt allein ca. 2 Stunden; ohne Checkpointing wuerde ein
# erneuter Verbindungsabbruch/Kernel-Tod alles davon zunichtemachen.
# Reimplementiert qpu.compute_posterior_predictive_probabilities lokal
# (identische Rechenlogik), speichert aber jede fertige Sample-
# Wahrscheinlichkeitsmatrix sofort einzeln ab und ueberspringt beim
# Neustart bereits vorhandene Samples.
# ============================================================

n_eval_examples = int(y_eval.shape[0])
n_samples_total_eval = int(samples_host.shape[0])

predictive_probs_dir = RESULT_DIR / f"{RUN_NAME}_predprob_cache"
predictive_probs_dir.mkdir(parents=True, exist_ok=True)

print("Evaluation examples:", n_eval_examples)
print("Evaluation batch size:", EVAL_BATCH_SIZE)
print("Posterior samples:", n_samples_total_eval)

sample_probabilities_list = []

for sample_index in range(n_samples_total_eval):
    sample_cache_path = predictive_probs_dir / f"sample_{sample_index:03d}.npy"

    if sample_cache_path.exists():
        probabilities_sample_host = np.load(sample_cache_path)
        print(f"Sample {sample_index + 1}/{n_samples_total_eval}: aus Zwischenspeicherung geladen")
    else:
        print(f"Evaluating posterior sample {sample_index + 1}/{n_samples_total_eval}")

        sample_theta = jnp.asarray(samples_host[sample_index])
        sample_params = rebuild_full_params(sample_theta)

        probability_batches = []

        for start in range(0, n_eval_examples, EVAL_BATCH_SIZE):
            end = min(start + EVAL_BATCH_SIZE, n_eval_examples)

            x_batch_eval = jax.tree.map(
                lambda array: array[start:end],
                x_eval,
            )

            rng_key, evaluation_key = random.split(rng_key)

            logits_batch, _ = env.s_model(
                sample_params,
                {},
                jnp.asarray(0.0, dtype=jnp.float32),
                x_batch_eval,
                train=False,
                key=evaluation_key,
            )

            probabilities_batch = jax.nn.softmax(logits_batch, axis=-1)
            probabilities_batch_host = np.asarray(jax.device_get(probabilities_batch))
            probability_batches.append(probabilities_batch_host)

            del x_batch_eval, logits_batch, probabilities_batch

        probabilities_sample_host = np.concatenate(probability_batches, axis=0)

        # Zwischenspeichern -- ueberlebt einen Kernel-Absturz / Verbindungsabbruch
        np.save(sample_cache_path, probabilities_sample_host)

        del sample_theta, sample_params, probability_batches
        gc.collect()

    sample_probabilities_list.append(probabilities_sample_host)

sample_probabilities = np.stack(sample_probabilities_list, axis=0)
mean_probabilities = np.mean(sample_probabilities, axis=0)

print("\nEvaluation completed")
print("Probability tensor shape:", sample_probabilities.shape)
print("Expected shape:", (n_samples_total_eval, n_eval_examples, mean_probabilities.shape[-1]))
print("All probabilities finite:", np.isfinite(sample_probabilities).all())


Evaluation examples: 7600
Evaluation batch size: 64
Posterior samples: 30
Evaluating posterior sample 1/30


Evaluating posterior sample 2/30


Evaluating posterior sample 3/30


Evaluating posterior sample 4/30


Evaluating posterior sample 5/30


...

Evaluating posterior sample 27/30


Evaluating posterior sample 28/30


Evaluating posterior sample 29/30


Evaluating posterior sample 30/30



Evaluation completed
Probability tensor shape: (30, 7600, 4)
Expected shape: (30, 7600, 4)
All probabilities finite: True


In [14]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


Accuracy: 0.9126315116882324
LPPD: -0.2564524710178375
Posterior predictive NLL: 0.2564524710178375
Brier Score: 0.13485465943813324
Mean predictive entropy: 0.2691551148891449
Mean expected entropy: 0.2667103111743927
Mean mutual information: 0.0024447976611554623


In [15]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


ECE: 0.005010363645851612


In [16]:
# ============================================================
# LOG POSTERIOR VALUES FOR ALL SAMPLES
# ============================================================

sample_log_posteriors_host = qpu.compute_sample_log_posteriors(
    qwen_log_posterior, sample_positions,
)


Computed log posterior values: 30
Range: -227169.71875 to -217957.75


In [17]:
# ============================================================
# SAVE RUN
# ============================================================

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

summary.update({
    "run_name": RUN_NAME,
    "method": "MILE",
    "sampler": "MCLMC",
    "variant": "multi-chain",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "chain_seeds": CHAIN_SEEDS,
    "n_chains": N_CHAINS,
    "n_samples_per_chain": N_SAMPLES_PER_CHAIN,
    "n_samples_total": int(samples_host.shape[0]),
    "posterior_key_seed": POSTERIOR_KEY_SEED,

    "n_per_class": N_PER_CLASS,
    "n_posterior_examples": N_POSTERIOR_EXAMPLES,
    "sequence_length": SEQ_LEN,

    "eval_mode": EVAL_MODE,
    "n_eval_examples": int(y_eval.shape[0]),

    "prior_std": PRIOR_STD,

    "warmup_steps": WARMUP_STEPS,
    "thinning_steps": THINNING_STEPS,
    "diagonal_preconditioning": DIAGONAL_PRECONDITIONING,
    "desired_energy_var_start": DESIRED_ENERGY_VAR_START,
    "desired_energy_var_end": DESIRED_ENERGY_VAR_END,
    "trust_in_estimate": TRUST_IN_ESTIMATE,
    "num_effective_samples": NUM_EFFECTIVE_SAMPLES,
    "step_size_init": STEP_SIZE_INIT,
    "adaptive_warmup": True,

    "tuned_parameters_per_chain": all_chain_tuned_parameters,

    "chain_durations_seconds": chain_durations_seconds,
    "chains_total_duration_seconds": chains_total_duration,
    "notebook_total_duration_seconds": time.time() - NOTEBOOK_START_TIME,

    "all_samples_finite": bool(np.isfinite(samples_host).all()),
    "n_parameter_dimensions": int(samples_host.shape[-1]),

    "minimum_distance_from_map": float(distances_from_map.min()),
    "mean_distance_from_map": float(distances_from_map.mean()),
    "maximum_distance_from_map": float(distances_from_map.max()),

    "minimum_log_posterior": float(sample_log_posteriors_host.min()),
    "mean_log_posterior": float(sample_log_posteriors_host.mean()),
    "maximum_log_posterior": float(sample_log_posteriors_host.max()),
})

metadata_arrays.update({
    "subset_indices": np.asarray(subset_indices_host),
    "labels": np.asarray(jax.device_get(y_eval)),
    "distances_from_map": np.asarray(distances_from_map),
    "log_posteriors": sample_log_posteriors_host,
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_positions,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results (Multi-Chain MILE, K=" + str(N_CHAINS) + "):")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
print()
print("Laufzeit pro Chain (s):", summary["chain_durations_seconds"])
print("Laufzeit alle Chains:  ", f'{summary["chains_total_duration_seconds"]:.1f}s'
      f' ({summary["chains_total_duration_seconds"]/60:.1f} min)')
print("Gesamtlaufzeit Notebook:", f'{summary["notebook_total_duration_seconds"]:.1f}s'
      f' ({summary["notebook_total_duration_seconds"]/60:.1f} min)')

print()
print("Zum Vergleich: Single-Chain-Ergebnis liegt bereits in")
print(RESULT_DIR, "unter einem anderen RUN_NAME.")



RUN SAVED
Run: mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval
Samples:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval_samples.npy
Probabilities: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval_probabilities.npy
Metadata:      /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval_metadata.npz
Summary:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/mile_qwen_agnews/mile_multichain_k1_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval_summary.json

Results (Multi-Chain MILE, K=1):
Accuracy:            0.9126315116882324
LPPD:                -0.2564524710178375
NLL:                 0.2564524710178375
Brier Score:         0.134854